# Media Bias — SBERT Embeddings & Similarity Analysis

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from src.embeddings import load_sbert, generate_embeddings, plot_tsne, plot_similarity_heatmap, nearest_neighbors

df = pd.read_csv('../data/sample_headlines.csv')
headlines = df['headline'].tolist()
labels = df['label'].tolist()
print(f"Loaded {len(headlines)} headlines")

## 1. Generate SBERT Embeddings

In [ ]:
model = load_sbert('all-MiniLM-L6-v2')
embeddings = generate_embeddings(headlines, model)
print(f"Embedding shape: {embeddings.shape}")  # (N, 384)

## 2. t-SNE Visualization

In [ ]:
plot_tsne(embeddings, labels, title="t-SNE of SBERT Embeddings — Media Bias")

## 3. Cosine Similarity Heatmap

In [ ]:
plot_similarity_heatmap(embeddings, labels, n=30)

## 4. Nearest Neighbor Analysis

In [ ]:
query_idx = 0
print(f"Query: '{headlines[query_idx]}' [{labels[query_idx]}]\n")
nn_df = nearest_neighbors(query_idx, embeddings, headlines, labels, top_k=5)
print(nn_df.to_string(index=False))

## 5. Intra-class vs Inter-class Similarity

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

label_ids = {'left': 0, 'neutral': 1, 'right': 2}
label_arr = np.array([label_ids[l] for l in labels])
sim_matrix = embeddings @ embeddings.T

intra, inter = [], []
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        if label_arr[i] == label_arr[j]:
            intra.append(sim_matrix[i, j])
        else:
            inter.append(sim_matrix[i, j])

plt.figure(figsize=(8, 5))
plt.hist(intra, bins=20, alpha=0.6, label='Same class (intra)', color='steelblue')
plt.hist(inter, bins=20, alpha=0.6, label='Different class (inter)', color='coral')
plt.title('Intra-class vs Inter-class Cosine Similarity', fontsize=13, fontweight='bold')
plt.xlabel('Cosine Similarity')
plt.legend()
plt.tight_layout()
plt.show()
print(f"Mean intra-class similarity : {np.mean(intra):.3f}")
print(f"Mean inter-class similarity : {np.mean(inter):.3f}")